# PASO Findings: P, A, S, and O

In [1]:
from __future__ import annotations

import json
import statistics
from collections import Counter, defaultdict
from pathlib import Path

import yaml


def stat_row(values: list[float]) -> dict[str, float | int]:
    ordered = sorted(float(v) for v in values)
    return {
        "count": len(ordered),
        "min": round(ordered[0], 3),
        "median": round(statistics.median(ordered), 3),
        "mean": round(statistics.fmean(ordered), 3),
        "max": round(ordered[-1], 3),
    }


def ms_to_minutes(value_ms: float) -> float:
    return round(float(value_ms) / 60000.0, 3)


def ms_to_seconds(value_ms: float) -> float:
    return round(float(value_ms) / 1000.0, 3)


def pct_delta(new_value: float, old_value: float) -> float:
    if not old_value:
        return 0.0
    return round(((float(new_value) - float(old_value)) / float(old_value)) * 100.0, 3)


def print_table(rows: list[dict], columns: list[str]) -> None:
    if not rows:
        print("<no rows>")
        return

    widths = {}
    for col in columns:
        widths[col] = max(len(col), max(len(str(row.get(col, ""))) for row in rows))

    header = " | ".join(col.ljust(widths[col]) for col in columns)
    divider = "-+-".join("-" * widths[col] for col in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(str(row.get(col, "")).ljust(widths[col]) for col in columns))


def collect_quality_means(profile_report: dict) -> dict[str, float]:
    raw_points = []
    final_points = []
    coverage = []
    point_count = []
    scan_count = []
    revisits = []
    pass1_elapsed = []
    pass2_elapsed = []
    for run in profile_report.get("runs", []):
        quality = run.get("result", {}).get("scan_quality", {})
        raw_points.append(float(quality.get("raw_points", 0.0)))
        final_points.append(float(quality.get("final_points", 0.0)))
        for item in quality.get("slice_analysis", []):
            coverage.append(float(item.get("coverage_percent", 0.0)))
            point_count.append(float(item.get("point_count", 0.0)))
            scan_count.append(float(item.get("scan_count", 0.0)))
        schedule = quality.get("schedule_experiment") or run.get("schedule_experiment_summary") or {}
        if schedule:
            revisits.append(float(schedule.get("selected_for_revisit_count", 0.0)))
            pass1_elapsed.append(float(schedule.get("pass1_elapsed_ms", 0.0)))
            pass2_elapsed.append(float(schedule.get("pass2_elapsed_ms", 0.0)))
    return {
        "raw_mean": round(statistics.fmean(raw_points), 3) if raw_points else 0.0,
        "final_mean": round(statistics.fmean(final_points), 3) if final_points else 0.0,
        "coverage_mean": round(statistics.fmean(coverage), 3) if coverage else 0.0,
        "point_count_mean": round(statistics.fmean(point_count), 3) if point_count else 0.0,
        "scan_count_mean": round(statistics.fmean(scan_count), 3) if scan_count else 0.0,
        "revisits_mean": round(statistics.fmean(revisits), 3) if revisits else 0.0,
        "pass1_mean_s": round(statistics.fmean(pass1_elapsed) / 1000.0, 3) if pass1_elapsed else 0.0,
        "pass2_mean_s": round(statistics.fmean(pass2_elapsed) / 1000.0, 3) if pass2_elapsed else 0.0,
    }


profile_dir = Path("../profiles")
robust_profile_paths = sorted(profile_dir.glob("profile_local_robust_3d*.json"))
if not robust_profile_paths:
    raise FileNotFoundError("No local robust_3d profiles found.")

baseline_profile_path = robust_profile_paths[0]
report = json.loads(baseline_profile_path.read_text())
runs = report["runs"]

optimization_profile_path = None
optimization_report = None
if len(robust_profile_paths) > 1:
    candidate_path = robust_profile_paths[-1]
    if candidate_path != baseline_profile_path:
        optimization_profile_path = candidate_path
        optimization_report = json.loads(optimization_profile_path.read_text())

schedule_profile_paths = sorted(profile_dir.glob("profile_local_schedule_experiment*.json"))
schedule_profile_path = schedule_profile_paths[-1] if schedule_profile_paths else None
schedule_report = json.loads(schedule_profile_path.read_text()) if schedule_profile_path else None

raw_config_path = Path(runs[0].get("config_path", "")) if runs else None
config_path = None
if raw_config_path:
    if raw_config_path.exists():
        config_path = raw_config_path
    else:
        fallback_config_path = (Path("..") / raw_config_path.name).resolve()
        if fallback_config_path.exists():
            config_path = fallback_config_path

config = {}
if config_path and config_path.exists():
    config = yaml.safe_load(config_path.read_text()) or {}

scan3d_config = config.get("scan3d", {})
plateau_iters = scan3d_config.get("plateau_iters")
plateau_tol = scan3d_config.get("plateau_tol")
min_scans = scan3d_config.get("min_scans")
if plateau_iters is not None and plateau_tol is not None and min_scans is not None:
    plateau_definition = (
        "A slice is marked as plateau once it has collected at least "
        f"{min_scans} full scans and the coverage change across the last "
        f"{plateau_iters} scans stays below {plateau_tol}."
    )
else:
    plateau_definition = (
        "A slice is marked as plateau when coverage stabilizes across recent scans "
        "after the minimum scan-count threshold has been reached."
    )

print(f"Baseline profile: {baseline_profile_path}")
print(f"Optimization profile: {optimization_profile_path}")
print(f"Schedule experiment profile: {schedule_profile_path}")
print(f"Baseline profiler version: {report.get('profiler_version')}")
print(f"Baseline schema version: {report.get('report_schema_version')}")
print(f"Baseline runs: {len(runs)}")
print(f"Resolved config path: {config_path}")
print(f"Plateau definition: {plateau_definition}")


Baseline profile: ../profiles/profile_local_robust_3d_20260326_112238.json
Optimization profile: ../profiles/profile_local_robust_3d_20260326_144623.json
Schedule experiment profile: ../profiles/profile_local_schedule_experiment_20260326_140637.json
Baseline profiler version: 2.0
Baseline schema version: 2
Baseline runs: 3
Resolved config path: /Users/zixu/Yr 2/Y2T2/Edge/project/RPLidar-3D-PointCloud/config_rpi.yaml
Plateau definition: A slice is marked as plateau once it has collected at least 10 full scans and the coverage change across the last 5 scans stays below 0.01.


## P: Profiling Findings

- Median end-to-end local `robust_3d` runtime is `367480.441 ms`, or about `6.125 minutes` per scan.
- `capture_robust_slice` is the dominant measured stage, averaging `214.694 s` per run and contributing about `58.535%` of total wall-clock runtime.
- `uninstrumented_wall_time` is the second-largest runtime block, averaging `148.173 s` per run and contributing about `40.398%` of total runtime.
- `init_lidar` contributes less than `1%`, while `servo_set_angle`, `voxel_downsample`, and `sor_filter` are negligible.
- The bottleneck is Pi-side slice acquisition plus surrounding wait and control-flow time, not servo commands or post-processing.


In [2]:
def summarize_stage_runtime(runs: list[dict]) -> list[dict]:
    total_run_ms = sum(float(run.get("elapsed_ms", 0.0)) for run in runs)
    stage_elapsed = defaultdict(list)
    for run in runs:
        for event in run.get("wrapped_events", []):
            name = event.get("name")
            elapsed_ms = event.get("elapsed_ms")
            if name and elapsed_ms is not None:
                stage_elapsed[name].append(float(elapsed_ms))

    rows = []
    instrumented_total_ms = 0.0
    for name, values in stage_elapsed.items():
        stats = stat_row(values)
        total_ms = round(sum(values), 3)
        instrumented_total_ms += total_ms
        rows.append(
            {
                "stage": name,
                "calls": len(values),
                "avg_per_run_s": ms_to_seconds(total_ms / len(runs)) if runs else 0.0,
                "share_percent": round((total_ms / total_run_ms) * 100.0, 3) if total_run_ms else 0.0,
                "median_call_ms": stats["median"],
                "max_call_ms": stats["max"],
            }
        )

    uninstrumented_ms = max(total_run_ms - instrumented_total_ms, 0.0)
    rows.append(
        {
            "stage": "uninstrumented_wall_time",
            "calls": len(runs),
            "avg_per_run_s": ms_to_seconds(uninstrumented_ms / len(runs)) if runs else 0.0,
            "share_percent": round((uninstrumented_ms / total_run_ms) * 100.0, 3) if total_run_ms else 0.0,
            "median_call_ms": "n/a",
            "max_call_ms": "n/a",
        }
    )
    return sorted(rows, key=lambda item: float(item["share_percent"]), reverse=True)


elapsed_summary = report.get("summary", {}).get("elapsed_ms", {})
cpu_summary = report.get("summary", {}).get("cpu_time_ms", {})
overall_rows = [
    {
        "metric": "elapsed_runtime",
        "count": elapsed_summary.get("count"),
        "median_ms": elapsed_summary.get("median_ms"),
        "median_min": ms_to_minutes(elapsed_summary.get("median_ms", 0.0)),
        "mean_min": ms_to_minutes(elapsed_summary.get("mean_ms", 0.0)),
        "max_min": ms_to_minutes(elapsed_summary.get("max_ms", 0.0)),
    },
    {
        "metric": "cpu_time",
        "count": cpu_summary.get("count"),
        "median_ms": cpu_summary.get("median_ms"),
        "median_min": ms_to_minutes(cpu_summary.get("median_ms", 0.0)),
        "mean_min": ms_to_minutes(cpu_summary.get("mean_ms", 0.0)),
        "max_min": ms_to_minutes(cpu_summary.get("max_ms", 0.0)),
    },
]

print("Overall runtime summary")
print_table(overall_rows, ["metric", "count", "median_ms", "median_min", "mean_min", "max_min"])

print("\nStage contribution summary")
print("Shares are against total wall-clock runtime across all runs.")
print("avg_per_run_s includes an explicit uninstrumented bucket for sleeps and other code outside wrapped stages.")
stage_rows = summarize_stage_runtime(runs)
print_table(stage_rows, ["stage", "calls", "avg_per_run_s", "share_percent", "median_call_ms", "max_call_ms"])


Overall runtime summary
metric          | count | median_ms  | median_min | mean_min | max_min
----------------+-------+------------+------------+----------+--------
elapsed_runtime | 3     | 367480.441 | 6.125      | 6.113    | 6.134  
cpu_time        | 3     | 10792.147  | 0.18       | 0.18     | 0.184  

Stage contribution summary
Shares are against total wall-clock runtime across all runs.
avg_per_run_s includes an explicit uninstrumented bucket for sleeps and other code outside wrapped stages.
stage                    | calls | avg_per_run_s | share_percent | median_call_ms | max_call_ms
-------------------------+-------+---------------+---------------+----------------+------------
capture_robust_slice     | 273   | 214.694       | 58.535        | 2306.462       | 3345.287   
uninstrumented_wall_time | 3     | 148.173       | 40.398        | n/a            | n/a        
init_lidar               | 3     | 3.515         | 0.958         | 3515.275       | 3515.277   
servo_set_angle 

## A: Analysis Findings

- All `273` slice samples ended by `plateau`; none ended by `max_scans`, retries, or I/O failure.
- Slice timing has a median of `2.306 s`, a mean of `2.359 s`, and a maximum of `3.345 s`.
- Slice scan count has a median of `10` and a mean of `11.106`, although some slices require up to `19` scans.
- Slice coverage remains in a narrow band, with a mean of `50.695%` and a range from `39.444%` to `56.944%`.
- Slice point count is reasonably stable, with a mean of `182.502` points and a range from `142` to `205`.
- The slowest slices do not consistently match the lowest coverage or lowest point counts, so extra runtime is more consistent with angle-dependent convergence behavior than obvious output sparsity.
- The slowest mean slice times appear at angles such as `168°`, `0°`, `80°`, `156°`, and `150°`, so convergence cost is at least partly angle-dependent.
- The dominant cost is normal convergence inside slice acquisition, not failure recovery or a hard `max_scans` limit.


In [3]:
slice_rows = []
for run_index, run in enumerate(runs, start=1):
    for item in run.get("result", {}).get("scan_quality", {}).get("slice_analysis", []):
        row = dict(item)
        row["run_index"] = run_index
        slice_rows.append(row)

print(f"Plateau definition used for interpretation: {plateau_definition}")
print(f"\nSlice samples collected: {len(slice_rows)}")

termination_counts = Counter(row.get("termination_reason", "unknown") for row in slice_rows)
termination_rows = [
    {"termination_reason": key, "count": value}
    for key, value in sorted(termination_counts.items(), key=lambda item: (-item[1], item[0]))
]
print("\nTermination reason counts")
print_table(termination_rows, ["termination_reason", "count"])

print("\nSlice timing stats (s)")
print(stat_row([row["elapsed_ms"] / 1000.0 for row in slice_rows]))

print("\nSlice scan-count stats")
print(stat_row([row["scan_count"] for row in slice_rows]))

print("\nSlice coverage stats (%)")
print(stat_row([row["coverage_percent"] for row in slice_rows]))

print("\nSlice point-count stats")
print(stat_row([row["point_count"] for row in slice_rows]))


Plateau definition used for interpretation: A slice is marked as plateau once it has collected at least 10 full scans and the coverage change across the last 5 scans stays below 0.01.

Slice samples collected: 273

Termination reason counts
termination_reason | count
-------------------+------
plateau            | 273  

Slice timing stats (s)
{'count': 273, 'min': 2.177, 'median': 2.306, 'mean': 2.359, 'max': 3.345}

Slice scan-count stats
{'count': 273, 'min': 10.0, 'median': 10.0, 'mean': 11.106, 'max': 19.0}

Slice coverage stats (%)
{'count': 273, 'min': 39.444, 'median': 51.111, 'mean': 50.695, 'max': 56.944}

Slice point-count stats
{'count': 273, 'min': 142.0, 'median': 184.0, 'mean': 182.502, 'max': 205.0}


In [4]:
slowest_slices = sorted(slice_rows, key=lambda row: float(row.get("elapsed_ms", 0.0)), reverse=True)[:10]
slowest_rows = [
    {
        "run": row["run_index"],
        "slice_index": row.get("slice_index"),
        "angle_deg": row.get("servo_angle_deg"),
        "elapsed_s": ms_to_seconds(row.get("elapsed_ms", 0.0)),
        "scan_count": row.get("scan_count"),
        "coverage_percent": row.get("coverage_percent"),
        "point_count": row.get("point_count"),
        "termination_reason": row.get("termination_reason"),
        "retry_count": row.get("retry_count", 0),
    }
    for row in slowest_slices
]
print("Slowest slices")
print_table(
    slowest_rows,
    [
        "run",
        "slice_index",
        "angle_deg",
        "elapsed_s",
        "scan_count",
        "coverage_percent",
        "point_count",
        "termination_reason",
        "retry_count",
    ],
)

angle_groups = defaultdict(list)
for row in slice_rows:
    angle_groups[row.get("servo_angle_deg")].append(row)

angle_rows = []
for angle, rows_for_angle in angle_groups.items():
    elapsed_values = [float(row["elapsed_ms"]) for row in rows_for_angle]
    scan_values = [float(row["scan_count"]) for row in rows_for_angle]
    angle_rows.append(
        {
            "angle_deg": angle,
            "samples": len(rows_for_angle),
            "mean_elapsed_s": ms_to_seconds(statistics.fmean(elapsed_values)),
            "max_elapsed_s": ms_to_seconds(max(elapsed_values)),
            "median_scan_count": round(statistics.median(scan_values), 3),
        }
    )

hot_angles = sorted(angle_rows, key=lambda row: row["mean_elapsed_s"], reverse=True)[:10]
print("\nSlowest angles by mean slice time")
print_table(hot_angles, ["angle_deg", "samples", "mean_elapsed_s", "max_elapsed_s", "median_scan_count"])

retry_total = sum(int(row.get("retry_count", 0)) for row in slice_rows)
io_error_count = sum(1 for row in slice_rows if row.get("had_io_error"))
print(f"\nTotal slice retries: {retry_total}")
print(f"Slices with I/O errors: {io_error_count}")


Slowest slices
run | slice_index | angle_deg | elapsed_s | scan_count | coverage_percent | point_count | termination_reason | retry_count
----+-------------+-----------+-----------+------------+------------------+-------------+--------------------+------------
3   | 0           | 0.0       | 3.345     | 19         | 53.889           | 194         | plateau            | 0          
2   | 84          | 168.0     | 3.298     | 18         | 51.389           | 185         | plateau            | 0          
3   | 78          | 156.0     | 3.289     | 19         | 48.889           | 176         | plateau            | 0          
1   | 40          | 80.0      | 3.186     | 18         | 56.389           | 203         | plateau            | 0          
2   | 74          | 148.0     | 3.17      | 18         | 46.389           | 167         | plateau            | 0          
2   | 90          | 180.0     | 3.051     | 16         | 53.611           | 193         | plateau            | 0          
3

## S: Scheduling Findings

- The tested two-pass adaptive revisit policy is slower than the baseline and should not be adopted as-is.
- Median scheduling runtime is `417356.635 ms`, or about `6.956 minutes` per scan, which is `13.572%` slower than baseline.
- The experiment revisits an average of `12` slices per run, with mean pass 1 time of `361.268 s` and mean pass 2 time of `49.248 s`.
- Quality moves slightly upward rather than downward: raw points `+1.096%`, final points `+1.229%`, mean coverage `+1.095%`, and mean slice point count `+1.096%`.
- The current scheduling policy does not solve the runtime problem because pass 1 already consumes almost the full baseline scan time, and pass 2 adds extra overhead on top.


In [5]:
if schedule_report is None:
    print("No schedule_experiment profile found.")
else:
    baseline_elapsed = float(report["summary"]["elapsed_ms"]["median_ms"])
    schedule_elapsed = float(schedule_report["summary"]["elapsed_ms"]["median_ms"])
    baseline_quality = collect_quality_means(report)
    schedule_quality = collect_quality_means(schedule_report)

    comparison_rows = [
        {
            "metric": "median_runtime_min",
            "baseline": ms_to_minutes(baseline_elapsed),
            "schedule_experiment": ms_to_minutes(schedule_elapsed),
            "delta_percent": pct_delta(schedule_elapsed, baseline_elapsed),
        },
        {
            "metric": "mean_raw_points",
            "baseline": baseline_quality["raw_mean"],
            "schedule_experiment": schedule_quality["raw_mean"],
            "delta_percent": pct_delta(schedule_quality["raw_mean"], baseline_quality["raw_mean"]),
        },
        {
            "metric": "mean_final_points",
            "baseline": baseline_quality["final_mean"],
            "schedule_experiment": schedule_quality["final_mean"],
            "delta_percent": pct_delta(schedule_quality["final_mean"], baseline_quality["final_mean"]),
        },
        {
            "metric": "mean_coverage_percent",
            "baseline": baseline_quality["coverage_mean"],
            "schedule_experiment": schedule_quality["coverage_mean"],
            "delta_percent": pct_delta(schedule_quality["coverage_mean"], baseline_quality["coverage_mean"]),
        },
        {
            "metric": "mean_slice_point_count",
            "baseline": baseline_quality["point_count_mean"],
            "schedule_experiment": schedule_quality["point_count_mean"],
            "delta_percent": pct_delta(schedule_quality["point_count_mean"], baseline_quality["point_count_mean"]),
        },
    ]
    print("Baseline vs scheduling experiment")
    print_table(comparison_rows, ["metric", "baseline", "schedule_experiment", "delta_percent"])

    schedule_rows = [
        {"metric": "mean_revisited_slices", "value": schedule_quality["revisits_mean"]},
        {"metric": "mean_pass1_time_s", "value": schedule_quality["pass1_mean_s"]},
        {"metric": "mean_pass2_time_s", "value": schedule_quality["pass2_mean_s"]},
    ]
    print("\nScheduling experiment summary")
    print_table(schedule_rows, ["metric", "value"])

    print("\nAdoption decision")
    print("Do not adopt current scheduling policy: runtime increased relative to baseline even though quality metrics improved slightly.")


Baseline vs scheduling experiment
metric                 | baseline  | schedule_experiment | delta_percent
-----------------------+-----------+---------------------+--------------
median_runtime_min     | 6.125     | 6.956               | 13.572       
mean_raw_points        | 16607.667 | 16789.667           | 1.096        
mean_final_points      | 15868.0   | 16063.0             | 1.229        
mean_coverage_percent  | 50.695    | 51.25               | 1.095        
mean_slice_point_count | 182.502   | 184.502             | 1.096        

Scheduling experiment summary
metric                | value  
----------------------+--------
mean_revisited_slices | 12.0   
mean_pass1_time_s     | 361.268
mean_pass2_time_s     | 49.248 

Adoption decision
Do not adopt current scheduling policy: runtime increased relative to baseline even though quality metrics improved slightly.


## O: Optimization Findings

- Lowering `min_scans` from `10` to `8` is a low-risk optimization candidate and can be adopted cautiously.
- Median optimized runtime is `363891.934 ms`, or about `6.065 minutes` per scan, which is `0.977%` faster than baseline.
- Mean slice scan count drops from `11.106` to `9.883` (`-11.012%`).
- Quality does not regress and instead improves slightly: raw points `+1.447%`, final points `+1.468%`, mean coverage `+1.448%`, and mean slice point count `+1.447%`.
- The total runtime gain is small because fixed waits and other non-capture costs still dominate the full scan.


In [6]:
if optimization_report is None:
    print("No separate optimization robust_3d profile found.")
else:
    baseline_elapsed = float(report["summary"]["elapsed_ms"]["median_ms"])
    optimized_elapsed = float(optimization_report["summary"]["elapsed_ms"]["median_ms"])
    baseline_quality = collect_quality_means(report)
    optimized_quality = collect_quality_means(optimization_report)

    comparison_rows = [
        {
            "metric": "median_runtime_min",
            "baseline": ms_to_minutes(baseline_elapsed),
            "optimized": ms_to_minutes(optimized_elapsed),
            "delta_percent": pct_delta(optimized_elapsed, baseline_elapsed),
        },
        {
            "metric": "mean_slice_scan_count",
            "baseline": baseline_quality["scan_count_mean"],
            "optimized": optimized_quality["scan_count_mean"],
            "delta_percent": pct_delta(optimized_quality["scan_count_mean"], baseline_quality["scan_count_mean"]),
        },
        {
            "metric": "mean_raw_points",
            "baseline": baseline_quality["raw_mean"],
            "optimized": optimized_quality["raw_mean"],
            "delta_percent": pct_delta(optimized_quality["raw_mean"], baseline_quality["raw_mean"]),
        },
        {
            "metric": "mean_final_points",
            "baseline": baseline_quality["final_mean"],
            "optimized": optimized_quality["final_mean"],
            "delta_percent": pct_delta(optimized_quality["final_mean"], baseline_quality["final_mean"]),
        },
        {
            "metric": "mean_coverage_percent",
            "baseline": baseline_quality["coverage_mean"],
            "optimized": optimized_quality["coverage_mean"],
            "delta_percent": pct_delta(optimized_quality["coverage_mean"], baseline_quality["coverage_mean"]),
        },
        {
            "metric": "mean_slice_point_count",
            "baseline": baseline_quality["point_count_mean"],
            "optimized": optimized_quality["point_count_mean"],
            "delta_percent": pct_delta(optimized_quality["point_count_mean"], baseline_quality["point_count_mean"]),
        },
    ]
    print("Baseline vs optimized min_scans=8")
    print_table(comparison_rows, ["metric", "baseline", "optimized", "delta_percent"])

    print("\nAdoption decision")
    print("Tentatively adopt min_scans=8 as a low-risk optimization: quality did not regress and runtime improved slightly, but the gain is small and should not be overstated.")


Baseline vs optimized min_scans=8
metric                 | baseline  | optimized | delta_percent
-----------------------+-----------+-----------+--------------
median_runtime_min     | 6.125     | 6.065     | -0.977       
mean_slice_scan_count  | 11.106    | 9.883     | -11.012      
mean_raw_points        | 16607.667 | 16848.0   | 1.447        
mean_final_points      | 15868.0   | 16101.0   | 1.468        
mean_coverage_percent  | 50.695    | 51.429    | 1.448        
mean_slice_point_count | 182.502   | 185.143   | 1.447        

Adoption decision
Tentatively adopt min_scans=8 as a low-risk optimization: quality did not regress and runtime improved slightly, but the gain is small and should not be overstated.
